# Jane Street — 基线模型 (Ridge + CatBoost)

## 运行说明
- **GPU**: 需要 GPU (CatBoost GPU 训练)，建议 T4 或 P100
- **数据**: 使用 Kaggle 公开数据集 `mohamedsameh0410/jane-street-dataset`
- **预计耗时**: Ridge ~5min, CatBoost ~20-40min (GPU)

In [ ]:
!pip install -q polars catboost

In [ ]:
import polars as pl
import numpy as np
from catboost import CatBoostRegressor, Pool
from pathlib import Path
import time
import gc

## 1. 配置

In [ ]:
# ====== 数据路径 ======
# Kaggle 环境: 用 mohamedsameh0410/jane-street-dataset
DATA_DIR = Path("/kaggle/input/jane-street-dataset")

# 本地调试: 取消下面注释
# DATA_DIR = Path("../data/processed")

TARGET_COL = "responder_6"
WEIGHT_COL = "weight"
TRAIN_END_DATE = 1400  # 训练/验证分割点 (82%/18%)

# 特征列
FEATURE_COLS = [f"feature_{i:02d}" for i in range(79)]
CAT_FEATURES = ["feature_09", "feature_10", "feature_11"]  # 原生分类
CONTINUOUS_COLS = [c for c in FEATURE_COLS if c not in CAT_FEATURES]
LAG_COLS = [f"responder_{i}_lag_1" for i in range(9)]

ALL_FEAT_COLS = CONTINUOUS_COLS + CAT_FEATURES + LAG_COLS

# 分层填充策略 (基于 EDA)
STABLE_FFILL = ["feature_00", "feature_02", "feature_03", "feature_15",
                "feature_39", "feature_42", "feature_50", "feature_53"]
UNSTABLE_FILL_ZERO = ["feature_01", "feature_04"]

print(f"连续特征: {len(CONTINUOUS_COLS)}")
print(f"分类特征: {len(CAT_FEATURES)}")
print(f"Lag特征: {len(LAG_COLS)}")
print(f"总特征数: {len(ALL_FEAT_COLS)}")

## 2. 数据预处理

In [ ]:
def preprocess(df):
    """分层缺失值填充 + 排序"""
    df = df.sort(["symbol_id", "date_id", "time_id"])
    
    # 慢变量: 按 symbol ffill
    df = df.with_columns([
        pl.col(STABLE_FFILL)
        .fill_null(strategy="forward")
        .over("symbol_id")
    ])
    # 快变量: 填 0
    df = df.with_columns([pl.col(UNSTABLE_FILL_ZERO).fill_null(0)])
    # 其余: ffill
    other = [c for c in FEATURE_COLS if c not in STABLE_FFILL + UNSTABLE_FILL_ZERO + CAT_FEATURES]
    df = df.with_columns([
        pl.col(other).fill_null(strategy="forward").over("symbol_id")
    ])
    df = df.fill_null(0)
    return df

def generate_lags(df):
    """从 responder 列生成前一天的 lag"""
    resp_cols = [f"responder_{i}" for i in range(9)]
    daily = df.group_by(["symbol_id", "date_id"], maintain_order=True) \
              .agg(pl.col(resp_cols).last()) \
              .sort(["symbol_id", "date_id"])
    
    # shift(1) → 前一天的 lag
    lag_exprs = [pl.col(c).shift(1).over("symbol_id").alias(f"{c}_lag_1") 
                 for c in resp_cols]
    daily_lags = daily.with_columns(lag_exprs)
    lag_out_cols = ["symbol_id", "date_id"] + [f"{c}_lag_1" for c in resp_cols]
    
    df = df.join(daily_lags.select(lag_out_cols), 
                 on=["symbol_id", "date_id"], how="left")
    return df.fill_null(0)

def load_and_preprocess(data_dir, is_train=True):
    """加载原始数据 + 预处理"""
    train_dir = data_dir / "train.parquet"
    
    # 扫描所有分区
    if train_dir.exists():
        # 本地路径
        df = pl.scan_parquet(str(train_dir / "partition_id=*" / "*.parquet"))
    else:
        # Kaggle 单文件路径
        df = pl.scan_parquet(str(data_dir / "train.parquet"))
    
    # 过滤 + 预处理
    if is_train:
        df = df.filter(pl.col('date_id') <= TRAIN_END_DATE)
    else:
        df = df.filter(pl.col('date_id') > TRAIN_END_DATE)
    
    df = df.collect()
    df = preprocess(df)
    df = generate_lags(df)
    
    return df

print("✓ 预处理函数定义完成")

In [ ]:
print("加载训练集...")
t0 = time.time()
train_df = load_and_preprocess(DATA_DIR, is_train=True)
print(f"  训练集: {train_df.height:,} rows × {train_df.width} cols, {time.time()-t0:.0f}s")

print("加载验证集...")
t0 = time.time()
val_df = load_and_preprocess(DATA_DIR, is_train=False)
print(f"  验证集: {val_df.height:,} rows, {time.time()-t0:.0f}s")

## 3. 评估函数

In [ ]:
def weighted_r2(y_true, y_pred, w):
    num = np.sum(w * (y_true - y_pred) ** 2)
    den = np.sum(w * y_true ** 2)
    return 1 - num / (den + 1e-38)

# 基线: 预测全 0
y_val = val_df[TARGET_COL].to_numpy().astype(np.float64)
w_val = val_df[WEIGHT_COL].to_numpy().astype(np.float64)
val_sids = val_df['symbol_id'].to_numpy()
naive_r2 = weighted_r2(y_val, np.zeros_like(y_val), w_val)
print(f"预测全 0 的 R²: {naive_r2:.6f}")
print(f"(任何模型必须超过这个值)")

## 4. Ridge 回归基线 (闭式解)

In [ ]:
print("=" * 50)
print("Ridge 回归")
print("=" * 50)

def prepare_X_ridge(df):
    """提取特征矩阵 (标准化 + one-hot)"""
    X_parts = []
    for c in CONTINUOUS_COLS + LAG_COLS:
        vals = df[c].to_numpy().astype(np.float64)
        vals = np.nan_to_num(vals, nan=0)
        m, s = vals.mean(), vals.std()
        if s == 0: s = 1
        X_parts.append(((vals - m) / s).reshape(-1, 1))
    
    # One-hot 分类特征
    for cat_col in CAT_FEATURES:
        vals = df[cat_col].to_numpy()
        for uv in sorted(set(int(v) for v in vals if not np.isnan(v))):
            X_parts.append((vals == uv).astype(np.float64).reshape(-1, 1))
    
    return np.hstack(X_parts)

# 准备训练数据
print("准备特征矩阵...")
X_train_r = prepare_X_ridge(train_df)
y_train_r = train_df[TARGET_COL].to_numpy().astype(np.float64)
w_train_r = train_df[WEIGHT_COL].to_numpy().astype(np.float64)

# 分块计算 XᵀWX 和 XᵀWy
p = X_train_r.shape[1]
print(f"特征维度: {p}")

xtx = X_train_r.T @ (X_train_r * w_train_r[:, np.newaxis])
xty = (X_train_r * w_train_r[:, np.newaxis]).T @ y_train_r
wy2 = np.dot(w_train_r, y_train_r ** 2)

del train_df, X_train_r, y_train_r, w_train_r
gc.collect()

# 准备验证数据
X_val_r = prepare_X_ridge(val_df)

# λ 搜索
print("\nλ 搜索...")
best_lam, best_r2 = None, -np.inf
for lam in [0.1, 1.0, 10.0, 100.0, 1000.0, 5000.0]:
    try:
        beta = np.linalg.solve(xtx + lam * np.eye(p), xty)
        yp = X_val_r @ beta
        r2 = weighted_r2(y_val, yp, w_val)
        train_r2 = 1 - (beta @ xtx @ beta - 2 * beta @ xty + wy2) / wy2
        marker = " ←" if r2 > best_r2 else ""
        print(f"  λ={lam:8.1f}  train={train_r2:.6f}  val={r2:.6f}{marker}")
        if r2 > best_r2:
            best_r2 = r2
            best_lam = lam
    except:
        print(f"  λ={lam:8.1f}  奇异矩阵")

beta = np.linalg.solve(xtx + best_lam * np.eye(p), xty)
ridge_r2 = best_r2
print(f"\nRidge 最佳 R²: {ridge_r2:.6f}")

## 5. CatBoost (GPU)

In [ ]:
print("=" * 50)
print("CatBoost (GPU)")
print("=" * 50)

# 重新加载数据 (CatBoost 用 native cat features, 不需要 one-hot)
# Ridge 训练时把 train_df 删了, 需要重载
train_df = load_and_preprocess(DATA_DIR, is_train=True)

# 采样 1/4 加速训练 (保留足够样本量)
SAMPLE_RATE = 4
train_df = train_df.filter(pl.int_range(0, pl.len()) % SAMPLE_RATE == 0)
print(f"采样后训练集: {train_df.height:,} rows")

# 准备数据
X_train_cb = train_df[ALL_FEAT_COLS].to_pandas()
y_train_cb = train_df[TARGET_COL].to_numpy().astype(np.float64)
w_train_cb = train_df[WEIGHT_COL].to_numpy().astype(np.float64)

X_val_cb = val_df[ALL_FEAT_COLS].to_pandas()

# CatBoost 分类特征索引
cat_indices = [ALL_FEAT_COLS.index(c) for c in CAT_FEATURES]
print(f"分类特征索引: {cat_indices}")

del train_df
gc.collect()

# 构建 Pool
train_pool = Pool(X_train_cb, y_train_cb, weight=w_train_cb, cat_features=cat_indices)
val_pool = Pool(X_val_cb, y_val, weight=w_val, cat_features=cat_indices)

# GPU 训练
model = CatBoostRegressor(
    iterations=2000,
    learning_rate=0.03,
    depth=6,
    l2_leaf_reg=5,
    random_strength=1,
    bagging_temperature=0.5,
    od_type='Iter',
    od_wait=100,
    loss_function='RMSE',
    eval_metric='RMSE',
    random_seed=42,
    task_type='GPU',
    devices='0',
    verbose=200,
    allow_writing_files=False,
)

print("开始训练...")
t0 = time.time()
model.fit(train_pool, eval_set=val_pool, verbose_eval=200)
print(f"训练耗时: {(time.time()-t0)/60:.1f} min")

In [ ]:
# 评估
y_pred_cb = model.predict(X_val_cb)
cb_r2 = weighted_r2(y_val, y_pred_cb, w_val)
print(f"CatBoost 验证 R²: {cb_r2:.6f}")
print(f"  提升 vs Ridge: {cb_r2 - ridge_r2:+.6f}")
print(f"  提升 vs 全0:  {cb_r2 - naive_r2:+.6f}")

# 特征重要性
importances = model.get_feature_importance()
feat_imp = sorted(zip(ALL_FEAT_COLS, importances), key=lambda x: -x[1])
print("\nTop 15 特征:")
for name, imp in feat_imp[:15]:
    print(f"  {name:25s}: {imp:.4f}")

In [ ]:
# 各 symbol 对比
print("\n各 symbol R² (Ridge vs CatBoost):")
print(f"{'symbol':>10s}  {'Ridge':>10s}  {'CatBoost':>10s}  {'Δ':>10s}")
print("-" * 42)
y_pred_ridge = X_val_r @ beta

for sid in sorted(set(int(s) for s in val_sids)):
    mask = val_sids == sid
    if mask.sum() > 1000:
        r_r = weighted_r2(y_val[mask], y_pred_ridge[mask], w_val[mask])
        r_c = weighted_r2(y_val[mask], y_pred_cb[mask], w_val[mask])
        print(f"  symbol_{sid:02d}  {r_r:10.6f}  {r_c:10.6f}  {r_c-r_r:+10.6f}")

# 集成: 简单平均
y_pred_ensemble = (y_pred_ridge + y_pred_cb) / 2
ensemble_r2 = weighted_r2(y_val, y_pred_ensemble, w_val)
print(f"\n简单平均集成 R²: {ensemble_r2:.6f}")
print(f"  提升 vs CatBoost: {ensemble_r2 - cb_r2:+.6f}")

In [ ]:
# 最终汇总
print("\n" + "=" * 50)
print("最终结果汇总")
print("=" * 50)
print(f"  基线 (全0):     {naive_r2:.6f}")
print(f"  Ridge (线性):   {ridge_r2:.6f}  (Δ baseline: {ridge_r2-naive_r2:+.6f})")
print(f"  CatBoost (树):  {cb_r2:.6f}  (Δ baseline: {cb_r2-naive_r2:+.6f})")
print(f"  Ensemble (平均): {ensemble_r2:.6f}  (Δ baseline: {ensemble_r2-naive_r2:+.6f})")